In [71]:
import os
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import open3d as o3d

ROOT_DIR = os.path.abspath('')
EXP_DIR  = os.path.join(ROOT_DIR, 'experiments',
                         'geotransformer.faces.stage4.gse.k3.max.oacl.stage2.sinkhorn')

PLY_PATH = os.path.join(ROOT_DIR, 'UHM_generated_data', '0.ply')

REAL_SCANS = [
    ('2189',         os.path.join(EXP_DIR, '2189.npy')),
    ('plank_scaled', os.path.join(EXP_DIR, 'plank_scaled.npy')),
]
REAL_COLORS = ['tomato', 'darkorange']

pcd = o3d.io.read_point_cloud(PLY_PATH)
pts = np.asarray(pcd.points).astype(np.float32)
print(f'Points loaded: {pts.shape[0]}')
for i, ax in enumerate(['X', 'Y', 'Z']):
    print(f'  {ax}  [{pts[:, i].min():.4f},  {pts[:, i].max():.4f}]')

Points loaded: 71926
  X  [-0.9382,  0.9607]
  Y  [-1.3095,  1.1818]
  Z  [-1.3188,  0.6413]


In [72]:
def pcd_trace(pts, color, name, size=1.5, opacity=0.6):
    return go.Scatter3d(
        x=pts[:, 0], y=pts[:, 1], z=pts[:, 2],
        mode='markers',
        marker=dict(size=size, color=color, opacity=opacity),
        name=name,
    )

def show_cloud(traces, title, camera=None):
    fig = go.Figure(data=traces)
    fig.update_layout(
        title=title, height=550,
        scene=dict(aspectmode='data', camera=camera or {}),
        legend=dict(itemsizing='constant'),
        margin=dict(l=0, r=0, b=0, t=40),
    )
    fig.show()

def subsample(arr, n):
    if len(arr) == 0:
        return arr
    idx = np.random.choice(len(arr), min(n, len(arr)), replace=False)
    return arr[idx]

def rms(pts):
    c = pts.mean(0)
    return float(np.sqrt(np.mean(np.sum((pts - c) ** 2, axis=1))))

def span(pts):
    return pts.max(0) - pts.min(0)   # [dx, dy, dz]

def span_ratios(pts):
    s = span(pts)
    return s[0] / s[2], s[1] / s[2]  # X/Z, Y/Z

def do_crop(pts, x_min, x_max, y_min, y_max, z_min, z_max=np.inf, subsample_frac=1.0):
    mask = (
        (pts[:, 0] >= x_min) & (pts[:, 0] <= x_max) &
        (pts[:, 1] >= y_min) & (pts[:, 1] <= y_max) &
        (pts[:, 2] >= z_min) & (pts[:, 2] <= z_max)
    )
    cropped = pts[mask]
    if subsample_frac < 1.0 and len(cropped) > 0:
        n = max(1, int(len(cropped) * subsample_frac))
        cropped = cropped[np.random.choice(len(cropped), n, replace=False)]
    return cropped, pts[~mask]

In [73]:
# ── Full head reference views — use these to pick your crop bounds ─────────────
MAX_VIZ = 30_000
pts_viz = subsample(pts, MAX_VIZ)

show_cloud([pcd_trace(pts_viz, 'steelblue', 'full head')],
           'Full head — front view', camera=dict(eye=dict(x=0, y=0, z=2)))
show_cloud([pcd_trace(pts_viz, 'steelblue', 'full head')],
           'Full head — side view (pick Z min here)', camera=dict(eye=dict(x=2, y=0, z=0)))
show_cloud([pcd_trace(pts_viz, 'steelblue', 'full head')],
           'Full head — top view (pick Y bounds here)', camera=dict(eye=dict(x=0, y=2, z=0)))

In [74]:
# ── EDIT THESE BOUNDS ─────────────────────────────────────────────────────────
# These are the BASE bounds used for the preview below.
# The distribution cell will also randomize each by ± JITTER_PCT.

X_MIN_BASE = -0.85
X_MAX_BASE =  0.85   # ±30% jitter → [0.60, 1.10], clips face sometimes
Y_MIN_BASE = -0.5
Y_MAX_BASE =  1.1    # back to original
Z_MIN_BASE = -0.2    # was -0.1, more lateral face

JITTER_PCT = 0.35


Z_MAX_BASE = np.inf

SUBSAMPLE_FRAC = 0.25   # fraction of cropped points to keep (matches dataset gen)
N_SAMPLES      = 150    # number of random crops for the distribution histograms
# ─────────────────────────────────────────────────────────────────────────────

face_pts, rest_pts = do_crop(
    pts,
    X_MIN_BASE, X_MAX_BASE,
    Y_MIN_BASE, Y_MAX_BASE,
    Z_MIN_BASE, Z_MAX_BASE,
    subsample_frac=SUBSAMPLE_FRAC,
)

print(f'Kept: {len(face_pts)} pts  |  Removed: {len(rest_pts)} pts')
for i, ax in enumerate(['X', 'Y', 'Z']):
    print(f'  {ax}  [{face_pts[:, i].min():.3f}, {face_pts[:, i].max():.3f}]')
xz, yz = span_ratios(face_pts)
print(f'\nMetrics for this crop:')
print(f'  RMS   = {rms(face_pts):.4f}')
print(f'  X/Z   = {xz:.3f}   (horiz-to-depth)')
print(f'  Y/Z   = {yz:.3f}   (vert-to-depth)')

Kept: 11608 pts  |  Removed: 25494 pts
  X  [-0.635, 0.667]
  Y  [-0.500, 1.099]
  Z  [-0.199, 0.640]

Metrics for this crop:
  RMS   = 0.5005
  X/Z   = 1.552   (horiz-to-depth)
  Y/Z   = 1.907   (vert-to-depth)


In [75]:
# ── Preview: kept (orange) vs removed (grey) ──────────────────────────────────
MAX_VIZ_EACH = 15_000
traces = [pcd_trace(subsample(face_pts, MAX_VIZ_EACH), 'orange', 'face (kept)', size=2)]
if len(rest_pts) > 0:
    traces.append(pcd_trace(subsample(rest_pts, MAX_VIZ_EACH), 'lightgrey', 'removed', size=1, opacity=0.3))
show_cloud(traces, f'Crop preview — {len(face_pts)} pts kept')

In [76]:
# ── Distribution across N random crops with these base bounds ─────────────────
# Simulates what the training distribution will look like.
# Each crop independently jitters each finite bound by ±JITTER_PCT.

rng = np.random.default_rng()

def jitter(base, lo_frac, hi_frac, rng):
    """Randomly scale base by a uniform factor in [lo_frac, hi_frac]. Preserves sign."""
    if not np.isfinite(base):
        return base
    return base * rng.uniform(lo_frac, hi_frac)

lo, hi = 1 - JITTER_PCT, 1 + JITTER_PCT

sample_rms, sample_xz, sample_yz = [], [], []
for _ in range(N_SAMPLES):
    xmin = jitter(X_MIN_BASE, lo, hi, rng)
    xmax = jitter(X_MAX_BASE, lo, hi, rng)
    ymin = jitter(Y_MIN_BASE, lo, hi, rng)
    ymax = jitter(Y_MAX_BASE, lo, hi, rng)
    zmin = jitter(Z_MIN_BASE, lo, hi, rng)
    zmax = jitter(Z_MAX_BASE, lo, hi, rng) if np.isfinite(Z_MAX_BASE) else np.inf

    cropped, _ = do_crop(pts, xmin, xmax, ymin, ymax, zmin, zmax,
                         subsample_frac=SUBSAMPLE_FRAC)
    if len(cropped) < 10:
        continue
    sample_rms.append(rms(cropped))
    xz_v, yz_v = span_ratios(cropped)
    sample_xz.append(xz_v)
    sample_yz.append(yz_v)

sample_rms = np.array(sample_rms)
sample_xz  = np.array(sample_xz)
sample_yz  = np.array(sample_yz)

# ── Load real scan metrics ────────────────────────────────────────────────────
real_metrics = []
for name, path in REAL_SCANS:
    rpts = np.load(path)[:, :3].astype(np.float32)
    r    = rms(rpts)
    xz_v, yz_v = span_ratios(rpts)
    real_metrics.append((name, r, xz_v, yz_v))
    print(f'{name:<16}  RMS={r:.4f}  X/Z={xz_v:.3f}  Y/Z={yz_v:.3f}')

print(f'\nTraining ({N_SAMPLES} crops):')
print(f'  RMS  mean={sample_rms.mean():.4f}  std={sample_rms.std():.4f}  range=[{sample_rms.min():.4f}, {sample_rms.max():.4f}]')
print(f'  X/Z  mean={sample_xz.mean():.3f}   std={sample_xz.std():.3f}   range=[{sample_xz.min():.3f}, {sample_xz.max():.3f}]')
print(f'  Y/Z  mean={sample_yz.mean():.3f}   std={sample_yz.std():.3f}   range=[{sample_yz.min():.3f}, {sample_yz.max():.3f}]')

2189              RMS=0.3561  X/Z=1.861  Y/Z=2.035
plank_scaled      RMS=0.5698  X/Z=1.651  Y/Z=1.998

Training (150 crops):
  RMS  mean=0.4891  std=0.0221  range=[0.4272, 0.5343]
  X/Z  mean=1.529   std=0.068   range=[1.362, 1.652]
  Y/Z  mean=1.800   std=0.208   range=[1.305, 2.239]


In [77]:
# ── Histograms: training distribution vs real scans ────────────────────────────
metrics_list = [
    ('RMS (scale signal)',     sample_rms, [m[1] for m in real_metrics]),
    ('X/Z span ratio (width)', sample_xz,  [m[2] for m in real_metrics]),
    ('Y/Z span ratio (height)',sample_yz,  [m[3] for m in real_metrics]),
]

fig = make_subplots(rows=1, cols=3,
                    subplot_titles=[m[0] for m in metrics_list])

for col, (title, train_vals, real_vals) in enumerate(metrics_list, start=1):
    fig.add_trace(
        go.Histogram(x=train_vals, nbinsx=20,
                     marker_color='steelblue', opacity=0.75,
                     name='training crops' if col == 1 else None,
                     showlegend=(col == 1)),
        row=1, col=col
    )
    for (name, *_), val, color in zip(REAL_SCANS, real_vals, REAL_COLORS):
        fig.add_vline(
            x=val, line_color=color, line_width=2.5, line_dash='dash',
            annotation_text=name, annotation_position='top right',
            row=1, col=col
        )

fig.update_layout(
    height=420,
    title_text=(
        f'Training distribution ({N_SAMPLES} crops, ±{int(JITTER_PCT*100)}% jitter) vs real scans<br>'
        f'Base bounds: X=[{X_MIN_BASE},{X_MAX_BASE}]  Y=[{Y_MIN_BASE},{Y_MAX_BASE}]  Z>={Z_MIN_BASE}'
    ),
    barmode='overlay',
    legend=dict(x=0.01, y=0.99),
    margin=dict(l=40, r=20, t=80, b=40),
)
fig.show()

In [78]:
# ── Stretch range — must match generate_random_view() in dataset generator ─────
STRETCH_X = (0.75, 1.25)   # U(0.75, 1.25) symmetric around 1.0
STRETCH_Y = (0.75, 1.25)
STRETCH_Z = (1.0,  1.0)    # disabled

In [79]:
# ── Distribution with anisotropic stretch applied ─────────────────────────────
stretched_rms, stretched_xz, stretched_yz = [], [], []

for _ in range(N_SAMPLES):
    xmin = jitter(X_MIN_BASE, lo, hi, rng)
    xmax = jitter(X_MAX_BASE, lo, hi, rng)
    ymin = jitter(Y_MIN_BASE, lo, hi, rng)
    ymax = jitter(Y_MAX_BASE, lo, hi, rng)
    zmin = jitter(Z_MIN_BASE, lo, hi, rng)
    zmax = jitter(Z_MAX_BASE, lo, hi, rng) if np.isfinite(Z_MAX_BASE) else np.inf

    cropped, _ = do_crop(pts, xmin, xmax, ymin, ymax, zmin, zmax,
                         subsample_frac=SUBSAMPLE_FRAC)
    if len(cropped) < 10:
        continue

    sx = rng.uniform(*STRETCH_X)
    sy = rng.uniform(*STRETCH_Y)
    sz = rng.uniform(*STRETCH_Z)
    cropped = cropped * np.array([sx, sy, sz], dtype=np.float32)

    stretched_rms.append(rms(cropped))
    xz_v, yz_v = span_ratios(cropped)
    stretched_xz.append(xz_v)
    stretched_yz.append(yz_v)

stretched_rms = np.array(stretched_rms)
stretched_xz  = np.array(stretched_xz)
stretched_yz  = np.array(stretched_yz)

print(f'With stretch ({N_SAMPLES} samples):')
print(f'  RMS  mean={stretched_rms.mean():.4f}  std={stretched_rms.std():.4f}  range=[{stretched_rms.min():.4f}, {stretched_rms.max():.4f}]')
print(f'  X/Z  mean={stretched_xz.mean():.3f}   std={stretched_xz.std():.3f}   range=[{stretched_xz.min():.3f}, {stretched_xz.max():.3f}]')
print(f'  Y/Z  mean={stretched_yz.mean():.3f}   std={stretched_yz.std():.3f}   range=[{stretched_yz.min():.3f}, {stretched_yz.max():.3f}]')

With stretch (150 samples):
  RMS  mean=0.4852  std=0.0550  range=[0.3548, 0.6116]
  X/Z  mean=1.502   std=0.217   range=[1.088, 2.053]
  Y/Z  mean=1.789   std=0.340   range=[1.046, 2.561]


In [80]:
# ── Overlay: no-stretch (blue) vs with-stretch (green) vs real scans ──────────
metrics_list_s = [
    ('RMS (scale signal)',
     sample_rms, stretched_rms, [m[1] for m in real_metrics]),
    ('X/Z span ratio (width)',
     sample_xz,  stretched_xz,  [m[2] for m in real_metrics]),
    ('Y/Z span ratio (height)',
     sample_yz,  stretched_yz,  [m[3] for m in real_metrics]),
]

fig2 = make_subplots(rows=1, cols=3,
                     subplot_titles=[m[0] for m in metrics_list_s])

for col, (title, base_vals, stretch_vals, real_vals) in enumerate(metrics_list_s, start=1):
    fig2.add_trace(
        go.Histogram(x=base_vals, nbinsx=20,
                     marker_color='steelblue', opacity=0.6,
                     name='no stretch' if col == 1 else None,
                     showlegend=(col == 1)),
        row=1, col=col
    )
    fig2.add_trace(
        go.Histogram(x=stretch_vals, nbinsx=20,
                     marker_color='mediumseagreen', opacity=0.6,
                     name='with stretch' if col == 1 else None,
                     showlegend=(col == 1)),
        row=1, col=col
    )
    for (name, *_), val, color in zip(REAL_SCANS, real_vals, REAL_COLORS):
        fig2.add_vline(
            x=val, line_color=color, line_width=2.5, line_dash='dash',
            annotation_text=name, annotation_position='top right',
            row=1, col=col
        )

fig2.update_layout(
    height=420,
    title_text=(
        f'Effect of anisotropic stretch  X={STRETCH_X}  Y={STRETCH_Y}  Z={STRETCH_Z}<br>'
        f'Blue = no stretch   Green = with stretch   Dashed = real scans'
    ),
    barmode='overlay',
    legend=dict(x=0.01, y=0.99),
    margin=dict(l=40, r=20, t=80, b=40),
)
fig2.show()

In [81]:
# ── OLD stretch (0.85,1.30) vs NEW stretch (0.75,1.25) — everything else identical ──
OLD_STRETCH = (0.85, 1.30)
NEW_STRETCH = (0.75, 1.25)

rng_cmp = np.random.default_rng(42)

old_s_rms, old_s_xz, old_s_yz = [], [], []
new_s_rms, new_s_xz, new_s_yz = [], [], []

for _ in range(N_SAMPLES):
    xmin = jitter(X_MIN_BASE, lo, hi, rng_cmp)
    xmax = jitter(X_MAX_BASE, lo, hi, rng_cmp)
    ymin = jitter(Y_MIN_BASE, lo, hi, rng_cmp)
    ymax = jitter(Y_MAX_BASE, lo, hi, rng_cmp)
    zmin = jitter(Z_MIN_BASE, lo, hi, rng_cmp)

    cropped, _ = do_crop(pts, xmin, xmax, ymin, ymax, zmin, np.inf,
                         subsample_frac=SUBSAMPLE_FRAC)
    if len(cropped) < 10:
        continue

    sx_old = rng_cmp.uniform(*OLD_STRETCH)
    sy_old = rng_cmp.uniform(*OLD_STRETCH)
    old_pts = cropped * np.array([sx_old, sy_old, 1.0], dtype=np.float32)
    old_s_rms.append(rms(old_pts))
    xz_v, yz_v = span_ratios(old_pts)
    old_s_xz.append(xz_v); old_s_yz.append(yz_v)

    sx_new = rng_cmp.uniform(*NEW_STRETCH)
    sy_new = rng_cmp.uniform(*NEW_STRETCH)
    new_pts = cropped * np.array([sx_new, sy_new, 1.0], dtype=np.float32)
    new_s_rms.append(rms(new_pts))
    xz_v, yz_v = span_ratios(new_pts)
    new_s_xz.append(xz_v); new_s_yz.append(yz_v)

old_s_rms = np.array(old_s_rms); old_s_xz = np.array(old_s_xz); old_s_yz = np.array(old_s_yz)
new_s_rms = np.array(new_s_rms); new_s_xz = np.array(new_s_xz); new_s_yz = np.array(new_s_yz)

print(f'OLD stretch {OLD_STRETCH} (mean~{sum(OLD_STRETCH)/2:.2f}, biased toward expansion):')
print(f'  RMS  {old_s_rms.mean():.4f} ± {old_s_rms.std():.4f}')
print(f'  X/Z  {old_s_xz.mean():.3f} ± {old_s_xz.std():.3f}  range=[{old_s_xz.min():.3f}, {old_s_xz.max():.3f}]')
print(f'  Y/Z  {old_s_yz.mean():.3f} ± {old_s_yz.std():.3f}  range=[{old_s_yz.min():.3f}, {old_s_yz.max():.3f}]')
print(f'NEW stretch {NEW_STRETCH} (mean=1.0, symmetric):')
print(f'  RMS  {new_s_rms.mean():.4f} ± {new_s_rms.std():.4f}')
print(f'  X/Z  {new_s_xz.mean():.3f} ± {new_s_xz.std():.3f}  range=[{new_s_xz.min():.3f}, {new_s_xz.max():.3f}]')
print(f'  Y/Z  {new_s_yz.mean():.3f} ± {new_s_yz.std():.3f}  range=[{new_s_yz.min():.3f}, {new_s_yz.max():.3f}]')

metrics_cmp = [
    ('RMS',               old_s_rms, new_s_rms, [m[1] for m in real_metrics]),
    ('X/Z (width/depth)', old_s_xz,  new_s_xz,  [m[2] for m in real_metrics]),
    ('Y/Z (height/depth)',old_s_yz,  new_s_yz,  [m[3] for m in real_metrics]),
]
fig_cmp = make_subplots(rows=1, cols=3, subplot_titles=[m[0] for m in metrics_cmp])
for col, (title, old_vals, new_vals, real_vals) in enumerate(metrics_cmp, start=1):
    fig_cmp.add_trace(go.Histogram(x=old_vals, nbinsx=20,
                                   marker_color='salmon', opacity=0.75,
                                   name=f'old stretch {OLD_STRETCH}' if col == 1 else None,
                                   showlegend=(col == 1)), row=1, col=col)
    fig_cmp.add_trace(go.Histogram(x=new_vals, nbinsx=20,
                                   marker_color='mediumseagreen', opacity=0.65,
                                   name=f'new stretch {NEW_STRETCH}' if col == 1 else None,
                                   showlegend=(col == 1)), row=1, col=col)
    for (name, *_), val, color in zip(REAL_SCANS, real_vals, REAL_COLORS):
        fig_cmp.add_vline(x=val, line_color=color, line_width=2.5, line_dash='dash',
                          annotation_text=name, annotation_position='top right',
                          row=1, col=col)

fig_cmp.update_layout(
    height=420,
    title_text=f'OLD stretch {OLD_STRETCH} (biased, mean~1.075)  vs  NEW stretch {NEW_STRETCH} (symmetric, mean=1.0)<br>Same bounds and jitter (±{int(JITTER_PCT*100)}%) in both',
    barmode='overlay', legend=dict(x=0.01, y=0.99),
    margin=dict(l=40, r=20, t=80, b=40),
)
fig_cmp.show()

OLD stretch (0.85, 1.3) (mean~1.07, biased toward expansion):
  RMS  0.5216 ± 0.0513
  X/Z  1.637 ± 0.203  range=[1.240, 2.099]
  Y/Z  1.937 ± 0.330  range=[1.279, 2.832]
NEW stretch (0.75, 1.25) (mean=1.0, symmetric):
  RMS  0.4899 ± 0.0479
  X/Z  1.515 ± 0.244  range=[1.102, 1.972]
  Y/Z  1.809 ± 0.339  range=[1.083, 2.752]
